In [1]:
#Установка библиотек
!pip install -q transformers evaluate accelerate
!pip install -q --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.6 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    BertConfig
)
import evaluate
from google.colab import userdata
from huggingface_hub import login

# Загрузка датасета RUSSE из RussianSuperGLUE
login(token=userdata.get('HF_TOKEN'))
model_name = "DeepPavlov/rubert-base-cased"

# Загрузка датасета из локальных jsonl файлов
dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "validation": "val.jsonl"
})

tokenizer = AutoTokenizer.from_pretrained(model_name)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [3]:
# Функция препроцессинга с явным выделением целевого слова
def preprocess_function(examples):
    first_sentences = [
        f"Слово: {w}. Контекст 1: {s1}"
        for w, s1 in zip(examples["word"], examples["sentence1"])
    ]
    second_sentences = [
        f"Контекст 2: {s2}"
        for s2 in examples["sentence2"]
    ]

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=128,
        padding=False
    )

    # Безопасное извлечение меток
    raw_labels = examples.get("label", examples.get("state", []))

    # конвертация любых форматов
    tokenized["label"] = [1 if str(l).lower() in ['true', '1'] else 0 for l in raw_labels]

    return tokenized

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

tokenized_dataset = tokenized_dataset.cast_column("label", Value("int64"))

# Фиксация формата колонок
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "token_type_ids", "attention_mask", "label"]
)



Map:   0%|          | 0/19845 [00:00<?, ? examples/s]

Map:   0%|          | 0/8505 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/19845 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8505 [00:00<?, ? examples/s]

In [4]:
# Метрика Accuracy
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)



In [5]:
# Инициализация модели


config = BertConfig.from_pretrained("DeepPavlov/rubert-base-cased", num_labels=2, problem_type="single_label_classification")
model = AutoModelForSequenceClassification.from_config(config) # Случайные веса




In [6]:
# Гиперпараметры обучения
training_args = TrainingArguments(
    output_dir="./rubert_russe_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=100,
    weight_decay=0.01,
    warmup_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none"
)



In [7]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Старт обучения
print("Начинаем обучение...")
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.672322,0.666120,0.630923
2,0.661173,0.664961,0.630923
3,0.664356,0.675106,0.630923
4,0.658396,0.661285,0.630923
5,0.652904,0.658464,0.630923


KeyboardInterrupt: 

In [ ]:

# Финальная оценка
eval_results = trainer.evaluate()
print("Validation Accuracy:", eval_results["eval_accuracy"])

Training Loss,Validation Loss,Epoch,Accuracy
0.651167,0.658563,4,0.630923


Validation Accuracy: 0.6309229864785421


In [ ]:
# Архивируем всю папку с результатами
!zip -r rubert_russe_results.zip /content/rubert_russe_results

# Запускаем скачивание архива
from google.colab import files
files.download('rubert_russe_results.zip')